# 🎙️ XTTS v2 Vietnamese Fine-Tuning - AUTO MODE

**Notebook này chạy tự động từ đầu đến cuối, không cần tương tác.**

## Cấu hình trước khi chạy:

### 1. Add Datasets (Settings → Add Data):
- `tinthnhphm21022004/data-speech-to-text` (audio files)
- `thanhphamtien2102224/weight-phowhisper` (manifest JSONL)

### 2. Kaggle Settings:
- **Accelerator**: GPU T4 x1
- **Internet**: ON
- **Persistence**: Files only (để save checkpoints)

### 3. Chạy notebook:
- Click **Run All** hoặc Shift+Enter trên cell duy nhất
- Notebook sẽ chạy ngầm, bạn có thể đóng tab
- Quay lại sau để xem kết quả trong Output

---

## Output:
- Checkpoints: `/kaggle/working/output/checkpoints/`
- Audio samples: `/kaggle/working/output/samples/`
- Logs: `/kaggle/working/output/logs/`
- Final zip: `/kaggle/working/xtts_output.zip`

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
XTTS v2 Vietnamese Fine-Tuning - Fully Automated
Chạy toàn bộ pipeline từ setup đến training đến inference.
"""

import os
import sys
import json
import shutil
import subprocess
import importlib
from pathlib import Path

print('='*80)
print('🚀 XTTS v2 Vietnamese Fine-Tuning - AUTO MODE')
print('='*80)

# ══════════════════════════════════════════════════════════════════════════════
# STEP 0: Check environment
# ══════════════════════════════════════════════════════════════════════════════
print('\n[STEP 0] Checking environment...')
import numpy as np
import torch

print(f'Python: {sys.version.split()[0]}')
print(f'NumPy: {np.__version__}')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

# ══════════════════════════════════════════════════════════════════════════════
# STEP 1: Clone repository
# ══════════════════════════════════════════════════════════════════════════════
print('\n[STEP 1] Cloning repository...')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
else:
    print('⚠️  No GPU detected! Training will be very slow.')

# ══════════════════════════════════════════════════════════════════════════════
# STEP 1: Clone repository
# ══════════════════════════════════════════════════════════════════════════════
print('\n[STEP 1] Cloning repository...')
REPO_DIR = '/kaggle/working/finetuneXTTSv2'

# Remove old repo
if os.path.isdir(REPO_DIR):
    shutil.rmtree(REPO_DIR)
    print('🗑️  Removed old repository')

# Clear module cache
stale = [k for k in sys.modules if k.startswith('xtts_finetune')]
for k in stale:
    del sys.modules[k]
if stale:
    print(f'🗑️  Cleared {len(stale)} cached modules')

# Clone
subprocess.check_call(
    ['git', 'clone', 'https://github.com/thanhptks212k4/finetuneXTTSv2.git', REPO_DIR],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
print(f'✅ Repository cloned to {REPO_DIR}')

# Add to path
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
importlib.invalidate_caches()

# ══════════════════════════════════════════════════════════════════════════════
# STEP 2: Install dependencies
# ══════════════════════════════════════════════════════════════════════════════
print('\n[STEP 2] Installing dependencies...')
subprocess.check_call(
    [sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pip', 'setuptools', 'wheel'],
    stdout=subprocess.DEVNULL,
)
subprocess.check_call(
    [sys.executable, '-m', 'pip', 'install', '-q', 'git+https://github.com/idiap/coqui-ai-TTS.git'],
    stdout=subprocess.DEVNULL,
)
subprocess.check_call(
    [sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub', 'librosa', 'soundfile'],
    stdout=subprocess.DEVNULL,
)
# Ensure numpy stays at 1.26.4
subprocess.check_call(
    [sys.executable, '-m', 'pip', 'install', '-q', 'numpy==1.26.4'],
    stdout=subprocess.DEVNULL,
)

# Verify
import numpy as np
from TTS.api import TTS
import TTS as _tts_pkg
print(f'✅ numpy: {np.__version__}')
print(f'✅ TTS: {_tts_pkg.__version__}')

# ══════════════════════════════════════════════════════════════════════════════
# STEP 4: Setup paths and fix manifests
# ══════════════════════════════════════════════════════════════════════════════
print('\n[STEP 4] Setting up paths and fixing manifests...')

AUDIO_ROOT    = '/kaggle/input/datasets/tinthnhphm21022004/data-speech-to-text'
MANIFEST_ROOT = '/kaggle/input/datasets/thanhphamtien2102224/weight-phowhisper'
TRAIN_JSONL   = f'{MANIFEST_ROOT}/train_full_manifest.jsonl'
TEST_JSONL    = f'{MANIFEST_ROOT}/test_manifest.jsonl'

# Check paths
for p in [AUDIO_ROOT, TRAIN_JSONL, TEST_JSONL]:
    if not os.path.exists(p):
        raise FileNotFoundError(f'❌ Path not found: {p}\nMake sure datasets are added in Kaggle settings!')
print('✅ All dataset paths exist')

# Fix manifest paths
FIXED_TRAIN = '/kaggle/working/train_manifest.jsonl'
FIXED_TEST  = '/kaggle/working/test_manifest.jsonl'
OLD_PREFIX  = '/kaggle/input/data-speech-to-text'
NEW_PREFIX  = AUDIO_ROOT

def fix_manifest(src, dst):
    ok, missing = 0, 0
    with open(src, 'r', encoding='utf-8') as fin, \
         open(dst, 'w', encoding='utf-8') as fout:
        for line in fin:
            obj = json.loads(line.strip())
            obj['audio'] = obj['audio'].replace(OLD_PREFIX, NEW_PREFIX)
            if os.path.exists(obj['audio']):
                fout.write(json.dumps(obj, ensure_ascii=False) + '\n')
                ok += 1
            else:
                missing += 1
    return ok, missing

train_ok, train_miss = fix_manifest(TRAIN_JSONL, FIXED_TRAIN)
test_ok, test_miss = fix_manifest(TEST_JSONL, FIXED_TEST)
print(f'✅ Train: {train_ok:,} valid samples ({train_miss} missing)')
print(f'✅ Test: {test_ok:,} valid samples ({test_miss} missing)')

# Find reference audio
REFERENCE_WAV = None
for root, dirs, files in os.walk(AUDIO_ROOT):
    for f in sorted(files):
        if f.lower().endswith('.wav'):
            REFERENCE_WAV = os.path.join(root, f)
            break
    if REFERENCE_WAV:
        break

if not REFERENCE_WAV:
    raise FileNotFoundError('❌ No .wav file found for reference audio!')

import soundfile as sf
info = sf.info(REFERENCE_WAV)
dur = info.frames / info.samplerate
print(f'✅ Reference audio: {REFERENCE_WAV}')
print(f'   Duration: {dur:.2f}s | SR: {info.samplerate} Hz')

# ══════════════════════════════════════════════════════════════════════════════
# STEP 5: Configure training
# ══════════════════════════════════════════════════════════════════════════════
print('\n[STEP 5] Configuring training...')

from xtts_finetune.config import TrainingConfig
from xtts_finetune.utils import get_logger, set_seed

WORKING    = '/kaggle/working'
OUTPUT_DIR = f'{WORKING}/output'

config = TrainingConfig(
    # Model
    hf_repo_id     = 'coqui/XTTS-v2',
    base_model_dir = f'{WORKING}/base_model',

    # Data
    train_manifest  = FIXED_TRAIN,
    val_manifest    = FIXED_TEST,
    reference_audio = REFERENCE_WAV,
    audio_root_remap = None,
    patch_size = 5000,

    # Output
    output_dir     = OUTPUT_DIR,
    checkpoint_dir = f'{OUTPUT_DIR}/checkpoints',
    sample_dir     = f'{OUTPUT_DIR}/samples',
    log_dir        = f'{OUTPUT_DIR}/logs',

    # Training (T4 16GB optimized)
    batch_size       = 2,
    grad_accum_steps = 8,
    learning_rate    = 2e-5,
    epochs_per_patch = 1,

    # Memory optimization
    use_fp16               = True,
    gradient_checkpointing = True,
    freeze_encoder         = True,

    # Speaker
    speaker_mode = 'single',

    # Kaggle
    zip_checkpoints = True,
    num_workers     = 2,
    seed            = 42,
)

logger = get_logger('kaggle_auto', config.log_dir)
set_seed(42)

print(f'✅ Config ready')
print(f'   Effective batch size: {config.batch_size * config.grad_accum_steps}')
print(f'   Output dir: {OUTPUT_DIR}')

# ══════════════════════════════════════════════════════════════════════════════
# STEP 6: Load and validate dataset
# ══════════════════════════════════════════════════════════════════════════════
print('\n[STEP 6] Loading and validating dataset...')

from xtts_finetune.dataset import load_manifest, validate_and_filter

train_raw = load_manifest(config.train_manifest, logger)
val_raw = load_manifest(config.val_manifest, logger)

print(f'Loaded {len(train_raw):,} train samples (raw)')
print(f'Loaded {len(val_raw):,} val samples (raw)')

# Validate
train_samples = validate_and_filter(train_raw, config, logger)
val_samples = validate_and_filter(val_raw, config, logger)

print(f'✅ Train: {len(train_samples):,} valid samples')
print(f'✅ Val: {len(val_samples):,} valid samples')

if len(train_samples) == 0:
    raise RuntimeError('❌ No valid training samples!')

# ══════════════════════════════════════════════════════════════════════════════
# STEP 7: Download base model
# ══════════════════════════════════════════════════════════════════════════════
print('\n[STEP 7] Downloading base model from HuggingFace...')

from xtts_finetune.model_loader import download_base_model

download_base_model(config, logger)
print('✅ Base model ready')

# ══════════════════════════════════════════════════════════════════════════════
# STEP 8: Load model and configure
# ══════════════════════════════════════════════════════════════════════════════
print('\n[STEP 8] Loading XTTS model...')

from xtts_finetune.model_loader import (
    load_xtts_model,
    configure_trainable_params,
    enable_gradient_checkpointing,
    extract_speaker_embedding,
)
from xtts_finetune.utils import log_gpu_memory

model, xtts_config = load_xtts_model(config, logger)
model = configure_trainable_params(model, config, logger)
enable_gradient_checkpointing(model, logger)
log_gpu_memory(logger, 'after model load')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
speaker_embedding = extract_speaker_embedding(
    model, xtts_config, config.reference_audio, device, logger
)

if speaker_embedding is not None:
    print(f'✅ Speaker embedding: {speaker_embedding.shape}')
else:
    print('⚠️  Using default speaker embedding')

# ══════════════════════════════════════════════════════════════════════════════
# STEP 9: Train
# ══════════════════════════════════════════════════════════════════════════════
print('\n[STEP 9] Starting training...')
print('='*80)

from xtts_finetune.trainer import XTTSTrainer

trainer = XTTSTrainer(
    model             = model,
    xtts_config       = xtts_config,
    config            = config,
    speaker_embedding = speaker_embedding,
)

try:
    final_metrics = trainer.train(train_samples, val_samples)
    
    print('\n' + '='*80)
    print('🎉 TRAINING COMPLETED SUCCESSFULLY!')
    print('='*80)
    print(f'Best validation loss: {trainer.best_val_loss:.4f}')
    print(f'Final validation MCD: {final_metrics.get("val_mcd", "N/A")}')
    print(f'Total steps: {trainer.global_step}')
    
except Exception as e:
    print('\n' + '='*80)
    print('❌ TRAINING FAILED')
    print('='*80)
    print(f'Error: {type(e).__name__}: {e}')
    import traceback
    traceback.print_exc()
    raise

# ══════════════════════════════════════════════════════════════════════════════
# STEP 10: Inference test
# ══════════════════════════════════════════════════════════════════════════════
print('\n[STEP 10] Running inference test...')

from xtts_finetune.inference import run_inference

test_texts = [
    'Xin chào, đây là hệ thống chuyển văn bản thành giọng nói tiếng Việt.',
    'Hôm nay trời đẹp, tôi rất vui được gặp bạn.',
    'Công nghệ trí tuệ nhân tạo đang phát triển rất nhanh.',
]

for i, text in enumerate(test_texts):
    out_path = f'{OUTPUT_DIR}/test_{i+1:02d}.wav'
    try:
        run_inference(
            text            = text,
            reference_audio = config.reference_audio,
            output_path     = out_path,
            config          = config,
            logger          = logger,
        )
        print(f'✅ [{i+1}] Generated: {out_path}')
    except Exception as e:
        print(f'⚠️  [{i+1}] Inference failed: {e}')

# ══════════════════════════════════════════════════════════════════════════════
# STEP 11: Create output zip
# ══════════════════════════════════════════════════════════════════════════════
print('\n[STEP 11] Creating output archive...')

zip_path = '/kaggle/working/xtts_output'
shutil.make_archive(zip_path, 'zip', OUTPUT_DIR)

size_mb = os.path.getsize(f'{zip_path}.zip') / 1024**2
print(f'✅ Output archived: {zip_path}.zip ({size_mb:.1f} MB)')

# ══════════════════════════════════════════════════════════════════════════════
# FINAL SUMMARY
# ══════════════════════════════════════════════════════════════════════════════
print('\n' + '='*80)
print('✅ ALL STEPS COMPLETED SUCCESSFULLY!')
print('='*80)
print('\n📂 Output locations:')
print(f'   Checkpoints: {config.checkpoint_dir}')
print(f'   Samples: {config.sample_dir}')
print(f'   Logs: {config.log_dir}')
print(f'   Archive: {zip_path}.zip')
print('\n📥 Download:')
print('   Kaggle → Output tab → xtts_output.zip')
print('\n🎉 Training pipeline completed!')
print('='*80)